# Module 7: Agentic AI with LangChain and LangGraph
## Topics Covered
- LangGraph Stateful Workflows
- Agent Architectures & ReAct Agents
- Multi-Agent Systems (Supervisor Pattern)
- Agentic RAG Architecture
- Scalable AI Applications

**Real-world scenario**: A customer support system with stateful conversations, ticket routing, specialist agents, and human escalation.

In [ ]:
%pip install -q langgraph langchain langchain-openai langchain-community faiss-cpu

In [ ]:
import os, json
from typing import TypedDict, List, Optional
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-api-key-here')

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('Ready')

---
## 1. LangGraph Fundamentals

LangGraph models agent workflows as directed graphs:

```
Key concepts:
  State   → TypedDict flowing between nodes
  Nodes   → Python functions that transform state
  Edges   → Connections (fixed or conditional)
  Memory  → MemorySaver checkpoints state across calls
```

**Why not just LCEL chains?**  
LCEL chains are DAGs (no loops). LangGraph supports **cycles** — agents can retry, refine, and loop until a goal is met.

---
## 2. Ticket Triage Graph

In [ ]:
class TicketState(TypedDict):
    ticket_text: str
    category: Optional[str]
    priority: Optional[str]
    sentiment: Optional[str]
    response: Optional[str]
    escalated: bool

def classify_ticket(state: TicketState) -> TicketState:
    prompt = f"""Classify this support ticket. Return ONLY valid JSON:
{{"category":"billing|technical|shipping|general","priority":"low|medium|high|urgent","sentiment":"positive|neutral|negative|angry"}}
Ticket: {state['ticket_text']}"""
    result = llm.invoke([HumanMessage(content=prompt)])
    try:
        clean = result.content.strip().replace('```json','').replace('```','')
        data = json.loads(clean)
        return {**state, **data}
    except Exception:
        return {**state, 'category': 'general', 'priority': 'medium', 'sentiment': 'neutral'}

def generate_response(state: TicketState) -> TicketState:
    sys = f"You are a {state['category']} support agent. Sentiment: {state['sentiment']}. Be empathetic if negative/angry."
    resp = llm.invoke([SystemMessage(content=sys), HumanMessage(content=state['ticket_text'])])
    return {**state, 'response': resp.content, 'escalated': False}

def escalate_to_human(state: TicketState) -> TicketState:
    msg = f"[ESCALATED] Priority={state['priority']}. Sentiment={state['sentiment']}.\nA support manager will contact you within 30 minutes."
    return {**state, 'response': msg, 'escalated': True}

def route(state: TicketState) -> str:
    if state['priority'] == 'urgent' or state['sentiment'] == 'angry':
        return 'escalate'
    return 'respond'

wf = StateGraph(TicketState)
wf.add_node('classify', classify_ticket)
wf.add_node('respond', generate_response)
wf.add_node('escalate', escalate_to_human)
wf.add_edge(START, 'classify')
wf.add_conditional_edges('classify', route, {'respond': 'respond', 'escalate': 'escalate'})
wf.add_edge('respond', END)
wf.add_edge('escalate', END)
ticket_app = wf.compile()
print('Graph compiled')

In [ ]:
tickets = [
    'My order hasn\'t moved in 3 days. When will it arrive?',
    'I AM FURIOUS! Charged twice and nobody responds! This is FRAUD!',
    'URGENT: Production server down due to your outage. Losing $10k/hour!',
]
for t in tickets:
    init = {'ticket_text': t, 'category': None, 'priority': None, 'sentiment': None, 'response': None, 'escalated': False}
    out = ticket_app.invoke(init)
    flag = 'ESCALATED' if out['escalated'] else 'AUTO-RESOLVED'
    print(f'\nTicket: {t[:70]}')
    print(f'  [{out["priority"]} | {out["sentiment"]}] -> {flag}')
    print(f'  Response: {out["response"][:120]}...')

---
## 3. ReAct Agent (prebuilt)

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the product knowledge base for policy information."""
    kb = {
        'return': 'Returns accepted within 30 days. Electronics: 15 days. Software non-returnable.',
        'shipping': 'Standard: free over $50, 5-7 days. Express: $9.99, 2-3 days.',
        'warranty': '1-year electronics warranty. Extended warranty $29-$99.',
        'payment': 'Visa, MC, Amex, PayPal, Apple Pay, Affirm installments.',
    }
    for k, v in kb.items():
        if k in query.lower():
            return v
    return 'Please contact support@store.com for more details.'

@tool
def get_order_status(order_id: str) -> str:
    """Get the status of an order by order ID."""
    orders = {
        'ORD-001': 'Delivered on 2024-03-10',
        'ORD-002': 'Shipped, tracking FX123456789, ETA 2024-03-15',
        'ORD-003': 'Processing, ETA 2024-03-16',
        'ORD-004': 'Cancelled, refund $45.99 issued',
    }
    return orders.get(order_id.upper(), f'Order {order_id} not found')

@tool
def create_ticket(summary: str, priority: str = 'normal') -> str:
    """Create a support ticket with given summary and priority."""
    import random
    tid = f'TKT-{random.randint(10000,99999)}'
    eta = '4 hours' if priority == 'high' else '24 hours'
    return f'Ticket {tid} created ({priority} priority). Response within {eta}.'

react_agent = create_react_agent(
    llm, tools=[search_knowledge_base, get_order_status, create_ticket],
    state_modifier='You are a helpful e-commerce support agent. Always be thorough.'
)

queries = [
    'My order ORD-002 hasn\'t arrived. It\'s urgent. Can you create a high priority ticket?',
    'I want to return a laptop. What is the process and timeline?',
]
for q in queries:
    print(f'\nUser: {q}')
    result = react_agent.invoke({'messages': [HumanMessage(content=q)]})
    print(f'Agent: {result["messages"][-1].content}')

---
## 4. Stateful Memory with Checkpointing

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
stateful_agent = create_react_agent(
    llm, tools=[search_knowledge_base, get_order_status, create_ticket],
    checkpointer=memory,
    state_modifier='You are a support agent. Remember the full conversation context.'
)

# Same thread_id = same conversation session
cfg = {'configurable': {'thread_id': 'session-001'}}

turns = [
    'Hi, I am Sarah. I have a question about my order.',
    'The order number is ORD-002.',
    'When exactly will it arrive?',
    'Please create a high priority ticket if it is delayed further.',
]
print('=== Stateful Multi-Turn Conversation ===')
for msg in turns:
    print(f'User: {msg}')
    out = stateful_agent.invoke({'messages': [HumanMessage(content=msg)]}, config=cfg)
    print(f'Agent: {out["messages"][-1].content}\n')

---
## 5. Multi-Agent Supervisor Pattern

In [ ]:
class MultiAgentState(TypedDict):
    query: str
    next_agent: str
    billing_response: Optional[str]
    technical_response: Optional[str]
    shipping_response: Optional[str]
    final_response: str
    iteration: int

def supervisor(state: MultiAgentState) -> MultiAgentState:
    answered = [a for a in ['billing','technical','shipping'] if state.get(f'{a}_response')]
    prompt = f"""Route this query to ONE of: billing, technical, shipping, or END.
Already answered by: {answered}. Do not repeat an agent.
Reply with only the routing word.
Query: {state['query']}"""
    result = llm.invoke([HumanMessage(content=prompt)])
    nxt = result.content.strip().lower()
    if nxt in answered or nxt not in ['billing','technical','shipping','end']:
        nxt = 'end'
    return {**state, 'next_agent': nxt, 'iteration': state.get('iteration',0)+1}

def billing_agent(state: MultiAgentState) -> MultiAgentState:
    r = llm.invoke([SystemMessage(content='You are a billing specialist.'), HumanMessage(content=state['query'])])
    return {**state, 'billing_response': r.content}

def technical_agent(state: MultiAgentState) -> MultiAgentState:
    r = llm.invoke([SystemMessage(content='You are a technical support specialist.'), HumanMessage(content=state['query'])])
    return {**state, 'technical_response': r.content}

def shipping_agent(state: MultiAgentState) -> MultiAgentState:
    r = llm.invoke([SystemMessage(content='You are a shipping specialist.'), HumanMessage(content=state['query'])])
    return {**state, 'shipping_response': r.content}

def synthesize(state: MultiAgentState) -> MultiAgentState:
    parts = []
    for a in ['billing','technical','shipping']:
        if state.get(f'{a}_response'):
            parts.append(f'{a.upper()}: {state[f"{a}_response"]}')
    final = '\n\n'.join(parts) if parts else 'Please contact support@store.com'
    return {**state, 'final_response': final}

def router(state: MultiAgentState) -> str:
    return state.get('next_agent', 'end')

mag = StateGraph(MultiAgentState)
for name, fn in [('supervisor',supervisor),('billing',billing_agent),('technical',technical_agent),('shipping',shipping_agent),('synthesize',synthesize)]:
    mag.add_node(name, fn)
mag.add_edge(START, 'supervisor')
mag.add_conditional_edges('supervisor', router, {'billing':'billing','technical':'technical','shipping':'shipping','end':'synthesize'})
for a in ['billing','technical','shipping']:
    mag.add_edge(a, 'supervisor')
mag.add_edge('synthesize', END)
multi_app = mag.compile()

# Test multi-agent with a complex query
query = 'I was charged twice for an order that arrived broken, and the tracking shows wrong address.'
result = multi_app.invoke({'query':query,'next_agent':'','billing_response':None,'technical_response':None,'shipping_response':None,'final_response':'','iteration':0})
agents_used = [a for a in ['billing','technical','shipping'] if result.get(f'{a}_response')]
print(f'Query: {query}')
print(f'Agents used: {agents_used}')
print(f'\nResponse:\n{result["final_response"][:500]}')

---
## 6. Agentic RAG (Self-Correcting Retrieval)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

docs = [
    Document(page_content='Returns accepted within 30 days. Electronics 15 days. Software non-returnable.'),
    Document(page_content='Shipping: free standard (5-7d) over $50. Express $9.99 (2-3d). Overnight $24.99.'),
    Document(page_content='Warranty: 1 year electronics. Extended warranty available $29-$99.'),
    Document(page_content='Payment: Visa, MC, Amex, PayPal, Apple Pay, Affirm installments for $100+.'),
]
vs = FAISS.from_documents(docs, OpenAIEmbeddings(model='text-embedding-3-small'))
retriever = vs.as_retriever(search_kwargs={'k': 2})

class RAGState(TypedDict):
    query: str
    retrieved: List[str]
    answer: str
    needs_more: bool
    iterations: int

def retrieve(state: RAGState) -> RAGState:
    found = retriever.invoke(state['query'])
    return {**state, 'retrieved': [d.page_content for d in found]}

def generate(state: RAGState) -> RAGState:
    context = '\n'.join(state['retrieved'])
    prompt = f"Answer using ONLY context. If insufficient, start with [NEED_MORE].\nContext: {context}\nQuestion: {state['query']}"
    ans = llm.invoke([HumanMessage(content=prompt)]).content
    needs = ans.startswith('[NEED_MORE]') and state.get('iterations',0) < 2
    return {**state, 'answer': ans, 'needs_more': needs, 'iterations': state.get('iterations',0)+1}

def refine_query(state: RAGState) -> RAGState:
    new_q = llm.invoke([HumanMessage(content=f'Rephrase more specifically: {state["query"]}')]).content
    return {**state, 'query': new_q}

def check_done(state: RAGState) -> str:
    return 'refine' if state['needs_more'] else END

rg = StateGraph(RAGState)
rg.add_node('retrieve', retrieve)
rg.add_node('generate', generate)
rg.add_node('refine', refine_query)
rg.add_edge(START, 'retrieve')
rg.add_edge('retrieve', 'generate')
rg.add_conditional_edges('generate', check_done, {'refine':'refine', END:END})
rg.add_edge('refine', 'retrieve')
rag_app = rg.compile()

r = rag_app.invoke({'query': 'What are your return and shipping policies?', 'retrieved':[], 'answer':'', 'needs_more':False, 'iterations':0})
print('Agentic RAG Answer:')
print(r['answer'])

---
## Summary

| Pattern | Key API |
|---------|--------|
| StateGraph | `StateGraph(TypedDict)` + `.add_node()` + `.add_edge()` |
| Conditional routing | `.add_conditional_edges()` |
| ReAct agent | `create_react_agent(llm, tools, checkpointer)` |
| Memory | `MemorySaver()` + `config={'thread_id': ...}` |
| Supervisor multi-agent | Nodes route back to supervisor until `END` |
| Agentic RAG | Retrieve → Generate → Refine loop |

**Next: Module 8 — CrewAI, AG2/AutoGen, and multi-agent frameworks**